In [6]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import plotly.io as pio
pio.templates.default = "plotly_white"

In [14]:
def plot_standardization_comparison(
    base_results_dir: str,
    model_name: str,
    fold_id: int,
    subject_id: str,
    trial_id: str,
    feature_name: str,
    data_source: str = "processed",
    processed_variant: str = "s1"
) -> go.Figure:
    """
    Plots an interactive comparison of standardization methods or raw sensor data.

    Args:
        base_results_dir: Path to the 'Results/Traditional_Standardization_Metrics' directory.
        model_name: Name of the model (e.g., 'RandomForest').
        fold_id: The fold number to load.
        subject_id: Subject ID to filter (e.g., 'J').
        trial_id: Trial ID to filter (e.g., '1').
        feature_name: The name of the feature to plot. If using processed data,
                      providing a base name (e.g., 'EEG_C3') will auto-append '_s1' or '_s2'.
        data_source: Either "processed" (shows train-only vs all-rows) or "raw" (shows pre-processing sensor data).
        processed_variant: If feature_name is a base name, choose "s1" or "s2" for processed data.

    Returns:
        A Plotly Figure object ready to be displayed in a Jupyter Notebook.
    """
    base_path = Path(base_results_dir)
    fold_dir = base_path / model_name / f"fold_{fold_id}"

    if not fold_dir.exists():
        raise FileNotFoundError(f"Fold directory not found: {fold_dir}")

    with open(fold_dir / "fold_metadata.json", "r") as f:
        metadata = json.load(f)

    # ---------------------------------------------------------------------
    # 1. PROCESSED DATA PIPELINE
    # ---------------------------------------------------------------------
    if data_source == "processed":
        std_data = np.load(fold_dir / "standardized_data.npz")
        time_data = np.load(fold_dir / "time_data.npz")
        label_data = np.load(fold_dir / "label_data.npz")
        trial_data = np.load(fold_dir / "trial_data.npz")

        feature_names = metadata["feature_names"]

        # Resolve feature name (handle base names by appending _s1 or _s2)
        if feature_name in feature_names:
            plot_feature_name = feature_name
        else:
            variant = f"_{processed_variant}"
            candidate = f"{feature_name}{variant}"
            if candidate in feature_names:
                plot_feature_name = candidate
            else:
                other_variant = "_s2" if processed_variant == "s1" else "_s1"
                candidate_other = f"{feature_name}{other_variant}"
                if candidate_other in feature_names:
                    plot_feature_name = candidate_other
                    print(f"Note: '{feature_name}' not found, using '{plot_feature_name}' instead.")
                else:
                    raise ValueError(f"Feature '{feature_name}' not found in processed features.")

        feat_idx = feature_names.index(plot_feature_name)

        # Filter by subject and trial
        subject_arr = trial_data["subject"].astype(str)
        trial_arr = trial_data["trial"].astype(str)
        mask = (subject_arr == str(subject_id)) & (trial_arr == str(trial_id))

        if not mask.any():
            raise ValueError(f"No data found for subject '{subject_id}', trial '{trial_id}' in fold {fold_id}.")

        time_filtered = time_data["time"][mask]
        y_gloc_filtered = label_data["y_gloc"][mask]
        imputed_filtered = label_data["imputed"][mask]
        train_only_filtered = std_data["train_only"][mask, feat_idx]
        all_rows_filtered = std_data["all_rows"][mask, feat_idx]

        fig = go.Figure()

        # Plot Train-Only (solid line, circles)
        fig.add_trace(go.Scatter(
            x=time_filtered, y=train_only_filtered,
            mode='lines',
            name='Train-Only Standardization',
            line=dict(color='royalblue', width=2),
            marker=dict(size=5, opacity=0.8, symbol='circle')
        ))

        # Plot All-Rows (dashed line, squares for overlap visibility)
        fig.add_trace(go.Scatter(
            x=time_filtered, y=all_rows_filtered,
            mode='lines',
            name='All-Rows (Legacy Leaky)',
            line=dict(color='crimson', width=2, dash='dot'),
            marker=dict(size=5, opacity=0.8, symbol='square')
        ))

        # Calculate Y bounds for marker placement
        y_max = np.nanmax(np.concatenate([train_only_filtered, all_rows_filtered]))
        y_min = np.nanmin(np.concatenate([train_only_filtered, all_rows_filtered]))
        y_range = y_max - y_min if y_max != y_min else 1.0

        # Label GLOC Events (1)
        gloc_idx = np.where(y_gloc_filtered == 1)[0]
        if len(gloc_idx) > 0:
            # Subtle background shading per window
            if len(time_filtered) > 1:
                width = np.median(np.diff(time_filtered[:10])) * 0.9
            else:
                width = 1.0

            for idx in gloc_idx:
                fig.add_vrect(
                    x0=time_filtered[idx] - width/2,
                    x1=time_filtered[idx] + width/2,
                    fillcolor="green", opacity=0.15, layer="below", line_width=0
                )
            # Star markers above the plot
            fig.add_trace(go.Scatter(
                x=time_filtered[gloc_idx],
                y=[y_max + y_range * 0.05] * len(gloc_idx),
                mode='markers',
                name='GLOC Event (1)',
                marker=dict(color='green', size=10, symbol='star'),
                showlegend=True
            ))

        # Label Imputed Points
        imputed_idx = np.where(imputed_filtered == 1)[0]
        if len(imputed_idx) > 0:
            fig.add_trace(go.Scatter(
                x=time_filtered[imputed_idx],
                y=train_only_filtered[imputed_idx],
                mode='markers',
                name='Imputed',
                marker=dict(color='orange', size=7, symbol='x', line=dict(width=1.5, color='black')),
                showlegend=True
            ))

        fig.update_layout(
            title=f"<b>Processed:</b> {plot_feature_name} | Subject: {subject_id} | Trial: {trial_id} | Fold: {fold_id}",
            xaxis_title="Time (s)",
            yaxis_title="Standardized Feature Value",
            hovermode="x unified",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(0,0,0,0)"),
            height=1200
        )
        return fig

    # ---------------------------------------------------------------------
    # 2. RAW DATA (PRE-PROCESSING)
    # ---------------------------------------------------------------------
    elif data_source == "raw":
        raw_data = np.load(fold_dir / "raw_per_sample_data.npz")
        raw_col_names = metadata["raw_column_names"]

        if feature_name not in raw_col_names:
            raise ValueError(f"Feature '{feature_name}' not found in raw columns. Available: {raw_col_names[:5]}...")

        feat_idx = raw_col_names.index(feature_name)

        subject_arr = raw_data["subject"].astype(str)
        trial_arr = raw_data["trial"].astype(str)
        mask = (subject_arr == str(subject_id)) & (trial_arr == str(trial_id))

        if not mask.any():
            raise ValueError(f"No raw data found for subject '{subject_id}', trial '{trial_id}' in fold {fold_id}.")

        time_filtered = raw_data["time"][mask]
        sensor_filtered = raw_data["sensor"][mask, feat_idx]

        # Map per-window GLOC/imputed labels to per-sample raw data using merge_asof
        time_data = np.load(fold_dir / "time_data.npz")
        label_data = np.load(fold_dir / "label_data.npz")
        trial_data_proc = np.load(fold_dir / "trial_data.npz")

        mask_proc = (trial_data_proc["subject"].astype(str) == str(subject_id)) & \
                    (trial_data_proc["trial"].astype(str) == str(trial_id))

        df_raw = pd.DataFrame({"time_raw": time_filtered, "sensor": sensor_filtered})
        df_proc = pd.DataFrame({
            "time_proc": time_data["time"][mask_proc],
            "y_gloc": label_data["y_gloc"][mask_proc],
            "imputed": label_data["imputed"][mask_proc]
        })

        # Sort by time for merge_asof
        df_raw = df_raw.sort_values("time_raw").reset_index(drop=True)
        df_proc = df_proc.sort_values("time_proc").reset_index(drop=True)

        # Assign the most recent window's label to each raw sample
        df_merged = pd.merge_asof(df_raw, df_proc, left_on="time_raw", right_on="time_proc", direction="backward")

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=df_merged["time_raw"], y=df_merged["sensor"],
            mode='lines',
            name='Raw Sensor Data',
            line=dict(color='purple', width=1)
        ))

        # Label GLOC Events (Mapped)
        gloc_mask = df_merged["y_gloc"] == 1
        if gloc_mask.any():
            y_max_raw = np.nanmax(df_merged["sensor"])
            y_min_raw = np.nanmin(df_merged["sensor"])
            y_range_raw = y_max_raw - y_min_raw if y_max_raw != y_min_raw else 1.0

            fig.add_trace(go.Scatter(
                x=df_merged.loc[gloc_mask, "time_raw"],
                y=[y_max_raw + y_range_raw * 0.05] * gloc_mask.sum(),
                mode='markers',
                name='GLOC Event (1) [Window-Mapped]',
                marker=dict(color='green', size=10, symbol='star')
            ))

        # Label Imputed Points (Mapped)
        imputed_mask = df_merged["imputed"] == 1
        if imputed_mask.any():
            fig.add_trace(go.Scatter(
                x=df_merged.loc[imputed_mask, "time_raw"],
                y=df_merged.loc[imputed_mask, "sensor"],
                mode='markers',
                name='Imputed [Window-Mapped]',
                marker=dict(color='orange', size=7, symbol='x', line=dict(width=1.5, color='black'))
            ))

        fig.update_layout(
            title=f"<b>Raw:</b> {feature_name} | Subject: {subject_id} | Trial: {trial_id} | Fold: {fold_id}",
            xaxis_title="Time (s)",
            yaxis_title="Raw Sensor Value",
            hovermode="x unified",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(0,0,0,0)"),
            height=1200
        )
        return fig

    else:
        raise ValueError("data_source must be either 'processed' or 'raw'")

In [15]:
# Define your parameters
RESULTS_DIR = "../../../Results/Traditional_Standardization_Metrics/Complete_Explicit"
MODEL = "KNN"  # e.g., "RandomForest"
FOLD = 0
SUBJECT = "11"            # Replace with actual subject ID from your data
TRIAL = "03"              # Replace with actual trial ID

# 1. Plot Processed Data (Train-Only vs All-Rows)
# If you know the exact processed name (e.g., "EEG_C3_s1"), use it directly.
# Otherwise, pass the base name and it will auto-resolve to _s1 or _s2.
fig_processed = plot_standardization_comparison(
    base_results_dir=RESULTS_DIR,
    model_name=MODEL,
    fold_id=FOLD,
    subject_id=SUBJECT,
    trial_id=TRIAL,
    feature_name="ECG Lead 2 - Equivital_v1_2derivative_range",
    data_source="processed",
    processed_variant="s1"
)
fig_processed.show()

In [16]:
# 2. Plot Raw Pre-Processing Data
# Shows the raw sensor stream, with GLOC/imputed labels intelligently mapped
# from the per-window data to the per-sample timeline.
fig_raw = plot_standardization_comparison(
    base_results_dir=RESULTS_DIR,
    model_name=MODEL,
    fold_id=FOLD,
    subject_id=SUBJECT,
    trial_id=TRIAL,
    feature_name="ECG Lead 2 - Equivital",       # Must match a name in raw_column_names
    data_source="raw"
)
fig_raw.show()